# Remapeo de Detecciones para Inclined_legs

## Objetivo
Este notebook implementa un sistema de **remapeo inteligente** para elementos tipo **Inclined_leg** (piernas inclinadas). 

### Contexto
Los Algoritmos Genéticos (AG) detectan daños en **nodos** de la estructura. Sin embargo, cuando hay daño en un **Inclined_leg**, es posible que el AG detecte el daño en uno de los **nodos adyacentes** (nodos_cercanos) pero no directamente en el elemento.

### Lógica de remapeo
Para cada Inclined_leg que **NO fue detectado directamente** (DeteccionOK = 0):
1. Verificamos si alguno de sus **nodos cercanos** (nodo_cercano_1 o nodo_cercano_2) tiene detección de daño
2. Si **SÍ** encuentra detección en algún nodo cercano → asignamos `DeteccionOK = 0.5` (detección parcial/indirecta)
3. Si **NO** encuentra nada → mantiene `DeteccionOK = 0` (sin detección)

### Entrada y Salida
- **Entrada**: `../jupyter_notebooks/todos_los_resultados_csv_remapeado.csv` (ya contiene beams remapeados)
- **Salida**: `../jupyter_notebooks/todos_los_resultados_csv_remapeado_final.csv` (con beams + inclined_legs remapeados)

---

## Pipeline de procesamiento

El proceso se divide en los siguientes bloques:
1. **Configuración**: Imports y definición de rutas
2. **Carga de mapeo**: Archivo `nodos_inclined_legs.csv` con relación elemento-nodos
3. **Carga de datos**: Archivo ya remapeado con beams
4. **Carga de resultados AG**: CSVs individuales por escenario de daño
5. **Función de verificación**: Busca detecciones en nodos cercanos
6. **Aplicación del remapeo**: Modifica DeteccionOK para inclined_legs con detección indirecta
7. **Conversión a numérico**: True→1, False→0, mantiene 0.5
8. **Guardado**: Exporta el dataframe final con ambos remapeos

---

## Bloque 1: Imports y Configuración de Rutas

Importamos las librerías necesarias y definimos las rutas de archivos.

In [10]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Rutas de archivos
MAPEO_NODOS = "nodos_inclined_legs.csv"
CSV_ENTRADA = "../jupyter_notebooks/todos_los_resultados_csv_remapeado.csv"
CSV_SALIDA = "../jupyter_notebooks/todos_los_resultados_csv_remapeado_final.csv"
CARPETA_CSVS = "salida_csvs"

print("✓ Configuración de rutas completada")
print(f"  - Mapeo de nodos: {MAPEO_NODOS}")
print(f"  - CSV entrada: {CSV_ENTRADA}")
print(f"  - CSV salida: {CSV_SALIDA}")
print(f"  - Carpeta con resultados AG: {CARPETA_CSVS}")

✓ Configuración de rutas completada
  - Mapeo de nodos: nodos_inclined_legs.csv
  - CSV entrada: ../jupyter_notebooks/todos_los_resultados_csv_remapeado.csv
  - CSV salida: ../jupyter_notebooks/todos_los_resultados_csv_remapeado_final.csv
  - Carpeta con resultados AG: salida_csvs


## Bloque 2: Cargar Mapeo de Nodos para Inclined_legs

Cargamos el archivo `nodos_inclined_legs.csv` que contiene la relación entre cada inclined_leg y sus nodos cercanos.

In [11]:
# Cargar el mapeo de nodos cercanos para inclined_legs
df_nodos = pd.read_csv(MAPEO_NODOS)

print("✓ Mapeo de nodos cargado correctamente")
print(f"  Total de inclined_legs mapeados: {len(df_nodos)}")
print(f"\nPrimeras filas del mapeo:")
print(df_nodos.head())
print(f"\nColumnas disponibles: {df_nodos.columns.tolist()}")

✓ Mapeo de nodos cargado correctamente
  Total de inclined_legs mapeados: 20

Primeras filas del mapeo:
   inclined_leg  nodo_i  nodo_j  nodo_cercano_1  nodo_cercano_2
0           112      35      43              38              39
1            97      36      44              39              40
2           102      33      41              40              37
3           107      34      42              38              37
4            87      28      36              32              31

Columnas disponibles: ['inclined_leg', 'nodo_i', 'nodo_j', 'nodo_cercano_1', 'nodo_cercano_2']


## Bloque 3: Cargar Datos Remapeados (con Beams)

Cargamos el archivo que ya contiene el remapeo de beams. Este será nuestro punto de partida para agregar el remapeo de inclined_legs.

In [12]:
# Cargar el archivo ya remapeado (contiene beams con valores 0, 0.5, y 1)
df_principal = pd.read_csv(CSV_ENTRADA)

print("✓ Datos principales cargados")
print(f"  Total de filas: {len(df_principal)}")
print(f"  Columnas: {df_principal.columns.tolist()}")

# Verificar que existen inclined_legs
if 'Type_of_element_to_search' in df_principal.columns:
    inclined_legs = df_principal[df_principal['Type_of_element_to_search'] == 'inclined_leg']
    print(f"\n  ✓ Inclined_legs encontrados: {len(inclined_legs)}")
    
    # Mostrar distribución actual de DetectionOK para inclined_legs
    print(f"\n  Distribución actual de DetectionOK en inclined_legs:")
    print(inclined_legs['DetectionOK'].value_counts().sort_index())
else:
    print("  ⚠️ Columna 'Type_of_element_to_search' no encontrada")

✓ Datos principales cargados
  Total de filas: 2160
  Columnas: ['ID', 'Element', 'Percentage', 'Time_seconds', 'Final_Objective', 'DetectionOK', 'AvgDispersion', 'StdDispersion', 'MeanAbsDispersion', 'N_FalsePositives', 'Type_of_element_to_search', 'Story']

  ✓ Inclined_legs encontrados: 360

  Distribución actual de DetectionOK en inclined_legs:
DetectionOK
0.0     72
1.0    288
Name: count, dtype: int64


## Bloque 4: Cargar Resultados Individuales del AG

Cargamos todos los archivos CSV individuales generados por el Algoritmo Genético (uno por cada escenario de daño).

In [13]:
# Cargar todos los CSVs de la carpeta salida_csvs/
archivos_csv = sorted([f for f in os.listdir(CARPETA_CSVS) if f.endswith('.csv')])

# Diccionario para almacenar cada CSV con su ID como clave
datos_ag = {}

for archivo in archivos_csv:
    ruta_completa = os.path.join(CARPETA_CSVS, archivo)
    # Extraer el ID del nombre del archivo (ej: ID_0001.csv -> 1)
    id_escenario = int(archivo.split('_')[1].split('.')[0])
    datos_ag[id_escenario] = pd.read_csv(ruta_completa)

print(f"✓ Cargados {len(datos_ag)} archivos CSV de resultados del AG")
print(f"  IDs: {sorted(datos_ag.keys())[:10]}... (mostrando primeros 10)")

# Verificar estructura de un CSV ejemplo
ejemplo_id = list(datos_ag.keys())[0]
print(f"\nEstructura del CSV ejemplo (ID {ejemplo_id}):")
print(f"  Columnas: {datos_ag[ejemplo_id].columns.tolist()}")
print(f"  Filas: {len(datos_ag[ejemplo_id])}")

✓ Cargados 2160 archivos CSV de resultados del AG
  IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]... (mostrando primeros 10)

Estructura del CSV ejemplo (ID 1):
  Columnas: ['Numero_de_nodo', 'Valor_de_daño_normalizado', 'Estado']
  Filas: 48


## Bloque 5: Función de Verificación de Detección en Nodos Cercanos

Esta función verifica si alguno de los nodos cercanos a un inclined_leg tiene detección de daño.

In [14]:
def verificar_deteccion_cercana(inclined_leg_id, id_escenario, df_nodos, datos_ag):
    """
    Verifica si un inclined_leg tiene detección de daño en alguno de sus nodos cercanos.
    
    Parámetros:
    -----------
    inclined_leg_id : int
        ID del elemento inclined_leg a verificar
    id_escenario : int
        ID del escenario/caso de daño
    df_nodos : DataFrame
        DataFrame con el mapeo de nodos cercanos
    datos_ag : dict
        Diccionario con los DataFrames de resultados del AG por escenario
    
    Retorna:
    --------
    bool : True si hay detección en algún nodo cercano, False en caso contrario
    """
    # Buscar los nodos cercanos de este inclined_leg
    fila_nodos = df_nodos[df_nodos['inclined_leg'] == inclined_leg_id]
    
    if fila_nodos.empty:
        return False
    
    # Obtener los nodos cercanos
    nodo_cercano_1 = fila_nodos.iloc[0]['nodo_cercano_1']
    nodo_cercano_2 = fila_nodos.iloc[0]['nodo_cercano_2']
    
    # Obtener el DataFrame correspondiente a este escenario
    if id_escenario not in datos_ag:
        return False
    
    df_escenario = datos_ag[id_escenario]
    
    # CORRECCIÓN: Los CSVs del AG usan 'Numero_de_nodo' y 'Estado' (no 'Elemento' y 'DeteccionOK')
    # Buscar en la columna 'Numero_de_nodo' los nodos cercanos
    deteccion_nodo1 = df_escenario[df_escenario['Numero_de_nodo'] == nodo_cercano_1]['Estado'].values
    deteccion_nodo2 = df_escenario[df_escenario['Numero_de_nodo'] == nodo_cercano_2]['Estado'].values
    
    # Verificar si alguno tiene detección (Estado == "Daño")
    tiene_deteccion_1 = len(deteccion_nodo1) > 0 and deteccion_nodo1[0] == 'Daño'
    tiene_deteccion_2 = len(deteccion_nodo2) > 0 and deteccion_nodo2[0] == 'Daño'
    
    return tiene_deteccion_1 or tiene_deteccion_2

print("✓ Función de verificación definida correctamente")

✓ Función de verificación definida correctamente


## Bloque 6: Aplicar Remapeo a Inclined_legs

Iteramos sobre cada fila del DataFrame y aplicamos el remapeo para inclined_legs que no tienen detección directa pero sí en nodos cercanos.

### 🔍 Diagnóstico Pre-Remapeo (Ejecutar antes del Bloque 6)

Verificamos la estructura de los datos antes de aplicar el remapeo.

In [15]:
# === DIAGNÓSTICO ===
print("🔍 Verificando estructura de datos antes del remapeo...\n")

# 1. Verificar columnas necesarias
print("1️⃣ Columnas en df_principal:")
print(df_principal.columns.tolist())

# 2. Verificar que existe la columna 'Element'
if 'Element' not in df_principal.columns:
    print("\n⚠️ ERROR: No existe la columna 'Element'")
    print("Columnas disponibles:", df_principal.columns.tolist())
else:
    print("\n✓ Columna 'Element' encontrada")

# 3. Verificar tipos de datos en DetectionOK
print("\n2️⃣ Tipos de datos en DetectionOK:")
print(df_principal['DetectionOK'].dtype)
print("\nValores únicos en DetectionOK:")
print(df_principal['DetectionOK'].unique())

# 4. Verificar un inclined_leg específico
inclined_sample = df_principal[df_principal['Type_of_element_to_search'] == 'inclined_leg'].head(1)
if not inclined_sample.empty:
    print("\n3️⃣ Ejemplo de inclined_leg:")
    print(inclined_sample[['ID', 'Element', 'Type_of_element_to_search', 'DetectionOK']].to_string())
    
    # Probar la función con este ejemplo
    test_id = inclined_sample.iloc[0]['Element']
    test_escenario = inclined_sample.iloc[0]['ID']
    print(f"\n4️⃣ Probando función con Element {test_id}, Escenario {test_escenario}:")
    try:
        resultado = verificar_deteccion_cercana(test_id, test_escenario, df_nodos, datos_ag)
        print(f"✓ Función ejecutada correctamente. Resultado: {resultado}")
    except Exception as e:
        print(f"❌ ERROR en la función: {e}")
        import traceback
        traceback.print_exc()

print("\n" + "="*60)
print("Diagnóstico completado. Revisa los resultados antes de continuar.")

🔍 Verificando estructura de datos antes del remapeo...

1️⃣ Columnas en df_principal:
['ID', 'Element', 'Percentage', 'Time_seconds', 'Final_Objective', 'DetectionOK', 'AvgDispersion', 'StdDispersion', 'MeanAbsDispersion', 'N_FalsePositives', 'Type_of_element_to_search', 'Story']

✓ Columna 'Element' encontrada

2️⃣ Tipos de datos en DetectionOK:
float64

Valores únicos en DetectionOK:
[1.  0.5 0. ]

3️⃣ Ejemplo de inclined_leg:
   ID  Element Type_of_element_to_search  DetectionOK
0   1        1              inclined_leg          1.0

4️⃣ Probando función con Element 1, Escenario 1:
✓ Función ejecutada correctamente. Resultado: False

Diagnóstico completado. Revisa los resultados antes de continuar.


In [16]:
# Crear una copia del DataFrame para modificar
df_modificado = df_principal.copy()

# Contadores para estadísticas
total_inclined_legs = 0
inclined_legs_remapeados = 0

print("Iniciando proceso de remapeo para inclined_legs...")
print("-" * 60)

# Iterar sobre cada fila del DataFrame
for idx, row in df_modificado.iterrows():
    # Verificar si es un inclined_leg
    if row['Type_of_element_to_search'] != 'inclined_leg':
        continue
    
    total_inclined_legs += 1
    
    # Verificar si NO tiene detección directa (DetectionOK == 0)
    if row['DetectionOK'] != 0:
        continue
    
    # Obtener información de la fila
    inclined_leg_id = row['Element']
    id_escenario = row['ID']
    
    # Verificar si hay detección en nodos cercanos
    if verificar_deteccion_cercana(inclined_leg_id, id_escenario, df_nodos, datos_ag):
        # Cambiar DetectionOK a 0.5 (detección parcial/indirecta)
        df_modificado.at[idx, 'DetectionOK'] = 0.5
        inclined_legs_remapeados += 1
        
        # Mostrar progreso cada 10 cambios
        if inclined_legs_remapeados % 10 == 0:
            print(f"  ✓ {inclined_legs_remapeados} inclined_legs remapeados...")

print("-" * 60)
print("✓ Proceso de remapeo completado")
print(f"\nEstadísticas:")
print(f"  - Total de inclined_legs procesados: {total_inclined_legs}")
print(f"  - Inclined_legs remapeados (0 → 0.5): {inclined_legs_remapeados}")
print(f"  - Porcentaje remapeado: {(inclined_legs_remapeados/total_inclined_legs*100) if total_inclined_legs > 0 else 0:.2f}%")

Iniciando proceso de remapeo para inclined_legs...
------------------------------------------------------------
  ✓ 10 inclined_legs remapeados...
  ✓ 20 inclined_legs remapeados...
  ✓ 30 inclined_legs remapeados...
  ✓ 40 inclined_legs remapeados...
  ✓ 50 inclined_legs remapeados...
  ✓ 60 inclined_legs remapeados...
  ✓ 70 inclined_legs remapeados...
------------------------------------------------------------
✓ Proceso de remapeo completado

Estadísticas:
  - Total de inclined_legs procesados: 360
  - Inclined_legs remapeados (0 → 0.5): 70
  - Porcentaje remapeado: 19.44%
------------------------------------------------------------
✓ Proceso de remapeo completado

Estadísticas:
  - Total de inclined_legs procesados: 360
  - Inclined_legs remapeados (0 → 0.5): 70
  - Porcentaje remapeado: 19.44%


## Bloque 7: Conversión Final de Valores Booleanos a Numéricos

Convertimos cualquier valor booleano restante (True/False) a numérico (1/0), manteniendo los valores 0.5 que ya fueron asignados.

In [17]:
# Ya no necesitamos convertir booleanos porque DetectionOK ya es numérico desde el principio
# Solo verificamos los valores finales

print("✓ Verificación de valores numéricos")
print("\nDistribución final de valores en DetectionOK:")
print(df_modificado['DetectionOK'].value_counts().sort_index())

# Verificar específicamente para inclined_legs
print("\nDistribución de DetectionOK para inclined_legs:")
inclined_final = df_modificado[df_modificado['Type_of_element_to_search'] == 'inclined_leg']
print(inclined_final['DetectionOK'].value_counts().sort_index())

✓ Verificación de valores numéricos

Distribución final de valores en DetectionOK:
DetectionOK
0.0      78
0.5     358
1.0    1724
Name: count, dtype: int64

Distribución de DetectionOK para inclined_legs:
DetectionOK
0.0      2
0.5     70
1.0    288
Name: count, dtype: int64


## Bloque 8: Guardar Archivo Final

Guardamos el DataFrame modificado con el remapeo completo (beams + inclined_legs) en el archivo de salida.

In [18]:
# Guardar el DataFrame modificado
df_modificado.to_csv(CSV_SALIDA, index=False)

print("=" * 70)
print("✅ PROCESO COMPLETADO EXITOSAMENTE")
print("=" * 70)
print(f"\nArchivo guardado en: {CSV_SALIDA}")
print(f"Total de filas: {len(df_modificado)}")
print(f"\nResumen por tipo de elemento:")
print(df_modificado.groupby('Type_of_element_to_search')['DetectionOK'].value_counts().sort_index())

print(f"\n{'='*70}")
print("Archivo listo para ser usado en el análisis de corrosión")
print("="*70)

✅ PROCESO COMPLETADO EXITOSAMENTE

Archivo guardado en: ../jupyter_notebooks/todos_los_resultados_csv_remapeado_final.csv
Total de filas: 2160

Resumen por tipo de elemento:
Type_of_element_to_search  DetectionOK
X_diagonal_brace           0.0               4
                           1.0            1436
beam                       0.0              72
                           0.5             288
inclined_leg               0.0               2
                           0.5              70
                           1.0             288
Name: count, dtype: int64

Archivo listo para ser usado en el análisis de corrosión


---

## ✅ Conclusión

Este notebook implementó exitosamente el **remapeo de detecciones para Inclined_legs**.

### Proceso realizado:

1. ✅ Se cargó el archivo con beams ya remapeados
2. ✅ Se identificaron todos los inclined_legs sin detección directa (DeteccionOK = 0)
3. ✅ Se verificó si alguno de sus nodos cercanos tenía detección de daño
4. ✅ Se asignó DeteccionOK = 0.5 a inclined_legs con detección indirecta
5. ✅ Se convirtieron todos los valores a formato numérico (0, 0.5, 1)
6. ✅ Se guardó el archivo final con ambos remapeos

### Valores en DeteccionOK:
- **0**: No se detectó daño (ni directa ni indirectamente)
- **0.5**: Detección indirecta (encontrado en nodos cercanos) - BEAMS e INCLINED_LEGS
- **1**: Detección directa en el elemento

### Próximo paso:
El archivo `todos_los_resultados_csv_remapeado_final.csv` ya puede ser usado en el notebook de análisis. 

**Importante:** Actualiza la ruta en `analisis_datos_abolladura.ipynb` para que lea:
```python
CSV_PATH = "todos_los_resultados_csv_remapeado_final.csv"
```

---

**Fecha de creación:** 29 de octubre, 2025  
**Propósito:** Remapeo de detecciones indirectas para inclined_legs basado en detección en nodos cercanos